In [0]:
%sql
SHOW TABLES IN default LIKE '%noaa%';

In [0]:
%sql
CREATE OR REPLACE TABLE default.noaa_stations_us_v2 AS
SELECT
  station AS station_id,
  first(latitude, true)  AS lat,
  first(longitude, true) AS lon
FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
WHERE station LIKE 'US%'
  AND latitude BETWEEN 24 AND 50
  AND longitude BETWEEN -125 AND -66
GROUP BY station;

In [0]:
%sql
SELECT COUNT(*) AS n_stations FROM default.noaa_stations_us_v2;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_nearest_stations_v2 AS
WITH dist AS (
  SELECT
    c.county_fips,
    s.station_id,
    6371 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s.lat - c.lat)/2), 2) +
      COS(RADIANS(c.lat)) * COS(RADIANS(s.lat)) *
      POWER(SIN(RADIANS(s.lon - c.lon)/2), 2)
    )) AS distance_km
  FROM default.county_centroids c
  CROSS JOIN default.noaa_stations_us_v2 s
)
SELECT county_fips, station_id, distance_km
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY county_fips ORDER BY distance_km) AS rn
  FROM dist
)
WHERE rn <= 5;

In [0]:
%sql
SELECT COUNT(*) AS rows_should_be_5x_counties FROM default.county_nearest_stations_v2;

SELECT * FROM default.county_nearest_stations_v2
WHERE county_fips='01001'
ORDER BY distance_km;

In [0]:
%sql
CREATE OR REPLACE TABLE default.station_month_metrics_clean_v2 AS
SELECT DISTINCT
  trim(station_id) AS station_id,
  cast(year as int) AS year,
  cast(month as int) AS month,
  cast(day_temperature_c as double) AS day_temperature_c,
  cast(night_temperature_c as double) AS night_temperature_c,
  cast(relative_humidity_pct as double) AS relative_humidity_pct,
  cast(monthly_rainfall_mm as double) AS monthly_rainfall_mm,
  cast(longest_frost_free_days as int) AS longest_frost_free_days,
  cast(corn_frost_free_ok as boolean) AS corn_frost_free_ok
FROM default.noaa_station_month_metrics
WHERE NOT (
  day_temperature_c IS NULL
  AND night_temperature_c IS NULL
  AND relative_humidity_pct IS NULL
  AND coalesce(monthly_rainfall_mm, 0) = 0
  AND coalesce(longest_frost_free_days, 0) = 0
  AND coalesce(corn_frost_free_ok, false) = false
);

In [0]:
%sql
CREATE OR REPLACE TABLE default.station_year_climate_v2 AS
SELECT
  station_id,
  year,

  -- Growing season months: Apr–Sep
  sum(CASE WHEN month BETWEEN 4 AND 9 THEN monthly_rainfall_mm END) AS rain_apr_sep_mm,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN day_temperature_c END)   AS daytemp_apr_sep_c,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN night_temperature_c END) AS nighttemp_apr_sep_c,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN relative_humidity_pct END) AS rh_apr_sep_pct,

  max(longest_frost_free_days) AS longest_frost_free_days,
  max(CASE WHEN corn_frost_free_ok THEN 1 ELSE 0 END) AS corn_frost_free_ok_int,

  -- Coverage (how many months reported)
  sum(CASE WHEN month BETWEEN 4 AND 9 THEN 1 ELSE 0 END) AS months_in_season,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND monthly_rainfall_mm IS NOT NULL THEN 1 ELSE 0 END) AS rain_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND day_temperature_c IS NOT NULL THEN 1 ELSE 0 END) AS daytemp_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND night_temperature_c IS NOT NULL THEN 1 ELSE 0 END) AS nighttemp_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND relative_humidity_pct IS NOT NULL THEN 1 ELSE 0 END) AS rh_months

FROM default.station_month_metrics_clean_v2
WHERE year BETWEEN 2010 AND 2024
GROUP BY station_id, year;

In [0]:
%sql
SELECT * FROM default.station_year_climate_v2 WHERE station_id='USW00014990' ORDER BY year LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_year_climate_v2 AS
WITH joined AS (
  SELECT
    n.county_fips,
    s.year,
    s.rain_apr_sep_mm,
    s.daytemp_apr_sep_c,
    s.nighttemp_apr_sep_c,
    s.rh_apr_sep_pct,
    s.longest_frost_free_days,
    s.corn_frost_free_ok_int,
    s.rain_months, s.daytemp_months, s.nighttemp_months, s.rh_months,
    1.0 / (n.distance_km + 1.0) AS wt
  FROM default.county_nearest_stations_v2 n
  JOIN default.station_year_climate_v2 s
    ON n.station_id = s.station_id
)
SELECT
  county_fips,
  year,

  -- weighted averages with null-safety
  sum(CASE WHEN rain_apr_sep_mm IS NOT NULL THEN rain_apr_sep_mm * wt END) / sum(CASE WHEN rain_apr_sep_mm IS NOT NULL THEN wt END) AS rain_apr_sep_mm,
  sum(CASE WHEN daytemp_apr_sep_c IS NOT NULL THEN daytemp_apr_sep_c * wt END) / sum(CASE WHEN daytemp_apr_sep_c IS NOT NULL THEN wt END) AS daytemp_apr_sep_c,
  sum(CASE WHEN nighttemp_apr_sep_c IS NOT NULL THEN nighttemp_apr_sep_c * wt END) / sum(CASE WHEN nighttemp_apr_sep_c IS NOT NULL THEN wt END) AS nighttemp_apr_sep_c,
  sum(CASE WHEN rh_apr_sep_pct IS NOT NULL THEN rh_apr_sep_pct * wt END) / sum(CASE WHEN rh_apr_sep_pct IS NOT NULL THEN wt END) AS rh_apr_sep_pct,

  -- these are basically station constants; take max
  max(longest_frost_free_days) AS longest_frost_free_days,
  max(corn_frost_free_ok_int)  AS corn_frost_free_ok_int,

  -- coverage totals
  sum(rain_months) AS rain_months_sum,
  sum(daytemp_months) AS daytemp_months_sum,
  sum(nighttemp_months) AS nighttemp_months_sum,
  sum(rh_months) AS rh_months_sum,

  count(*) AS stations_used
FROM joined
GROUP BY county_fips, year;

In [0]:
%sql
SELECT * FROM default.county_year_climate_v2
WHERE county_fips='01001'
ORDER BY year;

In [0]:
%sql
CREATE OR REPLACE TABLE default.yields_all_v2 AS
SELECT
  concat(
    lpad(cast(cast(`State Code` as int) as string), 2, '0'),
    lpad(cast(cast(`County Code` as int) as string), 3, '0')
  ) AS county_fips,
  trim(`State Name`) AS state_name,
  trim(`County Name`) AS county_name,
  cast(`Yield Year` as int) AS year,
  cast(`Yield Amount` as double) AS yield,
  trim(`Commodity Name`) AS commodity,
  trim(`Irrigation Practice Name`) AS irrigation
FROM default.rma_county_yields_report_399;

In [0]:
%sql
CREATE OR REPLACE TABLE default.gold_county_yield_climate_v2 AS
SELECT
  y.*,
  c.rain_apr_sep_mm,
  c.daytemp_apr_sep_c,
  c.nighttemp_apr_sep_c,
  c.rh_apr_sep_pct,
  c.longest_frost_free_days,
  c.corn_frost_free_ok_int,
  c.stations_used,
  c.rain_months_sum,
  c.daytemp_months_sum,
  c.nighttemp_months_sum,
  c.rh_months_sum
FROM default.yields_all_v2 y
LEFT JOIN default.county_year_climate_v2 c
  ON y.county_fips = c.county_fips
 AND y.year = c.year;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN rain_apr_sep_mm IS NULL THEN 1 ELSE 0 END) AS miss_rain,
  SUM(CASE WHEN daytemp_apr_sep_c IS NULL THEN 1 ELSE 0 END) AS miss_daytemp,
  SUM(CASE WHEN rh_apr_sep_pct IS NULL THEN 1 ELSE 0 END) AS miss_rh
FROM default.gold_county_yield_climate_v2;

In [0]:
%sql
SELECT year, COUNT(*) AS rows
FROM default.station_month_metrics_clean_v2
GROUP BY year
ORDER BY year;

In [0]:
%sql
CREATE OR REPLACE TABLE default.metrics_station_coords AS
SELECT
  m.station_id,
  s.lat,
  s.lon
FROM (SELECT DISTINCT station_id FROM default.station_month_metrics_clean_v2) m
JOIN default.noaa_stations_us_v2 s
  ON m.station_id = s.station_id;

In [0]:
%sql
CREATE OR REPLACE TABLE default.metrics_station_coords AS
WITH metric AS (
  SELECT DISTINCT upper(trim(station_id)) AS station_id
  FROM default.station_month_metrics_clean_v2
),
coords AS (
  SELECT
    upper(trim(station)) AS station_id,
    first(latitude, true)  AS lat,
    first(longitude, true) AS lon
  FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
  WHERE station LIKE 'US%'
  GROUP BY upper(trim(station))
)
SELECT
  m.station_id,
  c.lat,
  c.lon
FROM metric m
LEFT JOIN coords c
  ON m.station_id = c.station_id;

In [0]:
%sql
SELECT
  COUNT(*) AS metric_stations,
  SUM(CASE WHEN lat IS NULL OR lon IS NULL THEN 1 ELSE 0 END) AS missing_coords
FROM default.metrics_station_coords;

SELECT * FROM default.metrics_station_coords
WHERE lat IS NULL OR lon IS NULL
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_nearest_metricstations AS
WITH dist AS (
  SELECT
    c.county_fips,
    s.station_id,
    6371 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s.lat - c.lat)/2), 2) +
      COS(RADIANS(c.lat)) * COS(RADIANS(s.lat)) *
      POWER(SIN(RADIANS(s.lon - c.lon)/2), 2)
    )) AS distance_km
  FROM default.county_centroids c
  CROSS JOIN (
    SELECT * FROM default.metrics_station_coords
    WHERE lat IS NOT NULL AND lon IS NOT NULL
  ) s
)
SELECT county_fips, station_id, distance_km
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY county_fips ORDER BY distance_km) AS rn
  FROM dist
)
WHERE rn <= 5;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_year_climate AS
WITH joined AS (
  SELECT
    n.county_fips,
    s.year,
    s.rain_apr_sep_mm,
    s.daytemp_apr_sep_c,
    s.nighttemp_apr_sep_c,
    s.rh_apr_sep_pct,
    s.longest_frost_free_days,
    s.corn_frost_free_ok_int,
    1.0 / (n.distance_km + 1.0) AS wt
  FROM default.county_nearest_metricstations n
  JOIN default.station_year_climate_v2 s
    ON n.station_id = s.station_id
)
SELECT
  county_fips,
  year,
  sum(rain_apr_sep_mm * wt) / sum(wt) AS rain_apr_sep_mm,
  sum(daytemp_apr_sep_c * wt) / sum(wt) AS daytemp_apr_sep_c,
  sum(nighttemp_apr_sep_c * wt) / sum(wt) AS nighttemp_apr_sep_c,
  sum(rh_apr_sep_pct * wt) / sum(wt) AS rh_apr_sep_pct,
  max(longest_frost_free_days) AS longest_frost_free_days,
  max(corn_frost_free_ok_int) AS corn_frost_free_ok_int,
  count(*) AS stations_used
FROM joined
GROUP BY county_fips, year;

BLOCK


In [0]:
%sql
CREATE OR REPLACE TABLE default.station_coords AS
SELECT
  upper(trim(station)) AS station_id,
  first(latitude, true)  AS lat,
  first(longitude, true) AS lon
FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
WHERE station LIKE 'US%'
GROUP BY upper(trim(station));

In [0]:
%sql
CREATE OR REPLACE TABLE default.noaa_station_month_metrics_enriched AS
SELECT
  upper(trim(m.station_id)) AS station_id,
  m.state,
  m.station_name,
  cast(m.year as int) AS year,
  cast(m.month as int) AS month,
  cast(m.day_temperature_c as double) AS day_temperature_c,
  cast(m.night_temperature_c as double) AS night_temperature_c,
  cast(m.relative_humidity_pct as double) AS relative_humidity_pct,
  cast(m.monthly_rainfall_mm as double) AS monthly_rainfall_mm,
  cast(m.longest_frost_free_days as int) AS longest_frost_free_days,
  cast(m.corn_frost_free_ok as boolean) AS corn_frost_free_ok,
  c.lat,
  c.lon
FROM default.noaa_station_month_metrics m
LEFT JOIN default.station_coords c
  ON upper(trim(m.station_id)) = c.station_id;

In [0]:
%sql
SELECT
  COUNT(DISTINCT station_id) AS metric_stations,
  COUNT(DISTINCT CASE WHEN lat IS NOT NULL THEN station_id END) AS stations_with_coords
FROM default.noaa_station_month_metrics_enriched;

In [0]:
%sql
SELECT DISTINCT station_id
FROM default.noaa_station_month_metrics_enriched
WHERE lat IS NULL
LIMIT 50;

In [0]:
%sql
SELECT COUNT(*) AS n
FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
WHERE upper(regexp_replace(station, '[^A-Z0-9]', '')) = 'USW00014990';

FIX


In [0]:
%sql
-- Clean the monthly metrics table (remove “fake zero” rows)
CREATE OR REPLACE TABLE default.month_metrics_clean AS
SELECT DISTINCT
  upper(trim(station_id)) AS station_id,
  trim(state) AS state,
  trim(station_name) AS station_name,
  cast(year as int) AS year,
  cast(month as int) AS month,
  cast(day_temperature_c as double) AS day_temperature_c,
  cast(night_temperature_c as double) AS night_temperature_c,
  cast(relative_humidity_pct as double) AS relative_humidity_pct,
  cast(monthly_rainfall_mm as double) AS monthly_rainfall_mm,
  cast(longest_frost_free_days as int) AS longest_frost_free_days,
  cast(corn_frost_free_ok as boolean) AS corn_frost_free_ok,
  cast(latitude as double) AS lat,
  cast(longitude as double) AS lon,
  cast(elevation as double) AS elevation
FROM default.noaa_station_month_metrics
WHERE latitude IS NOT NULL AND longitude IS NOT NULL
  AND NOT (
    day_temperature_c IS NULL
    AND night_temperature_c IS NULL
    AND relative_humidity_pct IS NULL
    AND coalesce(monthly_rainfall_mm, 0) = 0
    AND coalesce(longest_frost_free_days, 0) = 0
    AND coalesce(corn_frost_free_ok, false) = false
  );

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT station_id) AS stations
FROM default.month_metrics_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE default.station_year_climate AS
SELECT
  station_id,
  year,
  first(lat, true) AS lat,
  first(lon, true) AS lon,

  -- Apr–Sep aggregate features
  sum(CASE WHEN month BETWEEN 4 AND 9 THEN monthly_rainfall_mm END) AS rain_apr_sep_mm,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN day_temperature_c END)   AS daytemp_apr_sep_c,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN night_temperature_c END) AS nighttemp_apr_sep_c,
  avg(CASE WHEN month BETWEEN 4 AND 9 THEN relative_humidity_pct END) AS rh_apr_sep_pct,

  max(longest_frost_free_days) AS longest_frost_free_days,
  max(CASE WHEN corn_frost_free_ok THEN 1 ELSE 0 END) AS corn_frost_free_ok_int,

  -- coverage counters (important!)
  sum(CASE WHEN month BETWEEN 4 AND 9 THEN 1 ELSE 0 END) AS months_in_season,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND monthly_rainfall_mm IS NOT NULL THEN 1 ELSE 0 END) AS rain_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND day_temperature_c IS NOT NULL THEN 1 ELSE 0 END) AS daytemp_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND night_temperature_c IS NOT NULL THEN 1 ELSE 0 END) AS nighttemp_months,
  sum(CASE WHEN month BETWEEN 4 AND 9 AND relative_humidity_pct IS NOT NULL THEN 1 ELSE 0 END) AS rh_months

FROM default.month_metrics_clean
WHERE year BETWEEN 2010 AND 2024
GROUP BY station_id, year;

In [0]:
%sql
SELECT * FROM default.station_year_climate
WHERE station_id='USW00014990'
ORDER BY year
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_nearest_metricstations AS
WITH stations AS (
  SELECT DISTINCT station_id, lat, lon
  FROM default.station_year_climate
),
dist AS (
  SELECT
    c.county_fips,
    s.station_id,
    6371 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s.lat - c.lat)/2), 2) +
      COS(RADIANS(c.lat)) * COS(RADIANS(s.lat)) *
      POWER(SIN(RADIANS(s.lon - c.lon)/2), 2)
    )) AS distance_km
  FROM default.county_centroids c
  CROSS JOIN stations s
)
SELECT county_fips, station_id, distance_km
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY county_fips ORDER BY distance_km) AS rn
  FROM dist
)
WHERE rn <= 5;

In [0]:
%sql
SELECT * FROM default.county_nearest_metricstations
WHERE county_fips='19061'  -- Dubuque County, IA
ORDER BY distance_km;

In [0]:
%sql
CREATE OR REPLACE TABLE default.county_year_climate AS
WITH joined AS (
  SELECT
    n.county_fips,
    s.year,
    s.rain_apr_sep_mm,
    s.daytemp_apr_sep_c,
    s.nighttemp_apr_sep_c,
    s.rh_apr_sep_pct,
    s.longest_frost_free_days,
    s.corn_frost_free_ok_int,
    s.rain_months, s.daytemp_months, s.nighttemp_months, s.rh_months,
    1.0 / (n.distance_km + 1.0) AS wt
  FROM default.county_nearest_metricstations n
  JOIN default.station_year_climate s
    ON n.station_id = s.station_id
)
SELECT
  county_fips,
  year,

  sum(CASE WHEN rain_apr_sep_mm IS NOT NULL THEN rain_apr_sep_mm * wt END)
    / sum(CASE WHEN rain_apr_sep_mm IS NOT NULL THEN wt END) AS rain_apr_sep_mm,

  sum(CASE WHEN daytemp_apr_sep_c IS NOT NULL THEN daytemp_apr_sep_c * wt END)
    / sum(CASE WHEN daytemp_apr_sep_c IS NOT NULL THEN wt END) AS daytemp_apr_sep_c,

  sum(CASE WHEN nighttemp_apr_sep_c IS NOT NULL THEN nighttemp_apr_sep_c * wt END)
    / sum(CASE WHEN nighttemp_apr_sep_c IS NOT NULL THEN wt END) AS nighttemp_apr_sep_c,

  sum(CASE WHEN rh_apr_sep_pct IS NOT NULL THEN rh_apr_sep_pct * wt END)
    / sum(CASE WHEN rh_apr_sep_pct IS NOT NULL THEN wt END) AS rh_apr_sep_pct,

  max(longest_frost_free_days) AS longest_frost_free_days,
  max(corn_frost_free_ok_int)  AS corn_frost_free_ok_int,

  sum(rain_months) AS rain_months_sum,
  sum(daytemp_months) AS daytemp_months_sum,
  sum(nighttemp_months) AS nighttemp_months_sum,
  sum(rh_months) AS rh_months_sum,

  count(*) AS stations_used
FROM joined
GROUP BY county_fips, year;

In [0]:
%sql
SELECT county_fips, year, rain_apr_sep_mm, daytemp_apr_sep_c, rh_apr_sep_pct, stations_used
FROM default.county_year_climate
WHERE county_fips='19061'
ORDER BY year;

In [0]:
%sql
CREATE OR REPLACE TABLE default.gold_county_yield_climate AS
SELECT
  y.*,
  c.rain_apr_sep_mm,
  c.daytemp_apr_sep_c,
  c.nighttemp_apr_sep_c,
  c.rh_apr_sep_pct,
  c.longest_frost_free_days,
  c.corn_frost_free_ok_int,
  c.stations_used,
  c.rain_months_sum,
  c.daytemp_months_sum,
  c.nighttemp_months_sum,
  c.rh_months_sum
FROM default.yields_all_v2 y
LEFT JOIN default.county_year_climate c
  ON y.county_fips = c.county_fips
 AND y.year = c.year;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN rain_apr_sep_mm IS NULL THEN 1 ELSE 0 END) AS miss_rain,
  SUM(CASE WHEN daytemp_apr_sep_c IS NULL THEN 1 ELSE 0 END) AS miss_daytemp,
  SUM(CASE WHEN rh_apr_sep_pct IS NULL THEN 1 ELSE 0 END) AS miss_rh
FROM default.gold_county_yield_climate;